In [ ]:
# youtube - https://www.youtube.com/watch?v=zCEJurLGFRk&t=14s
# % pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib gspread
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
from gspread_dataframe import set_with_dataframe

from data import Data
from cache_pandas import timed_lru_cache
import logging

@timed_lru_cache(seconds=None, maxsize=None)
def dataF():
    logging.info("Fetching data in dataF()")
    df = Data()
    vix_data = df.vix_history()
    policy_rate1, policy_rate2, policy_rate3 = df.policy_rate()
    fx = df.forex_exchange()
    cds = df.cds()
    liquidity = df.liquidity()
    gdp_growth = df.gdp_growth()
    logging.info("Completed fetching data in dataF()")
    return (
        vix_data, #use this
        policy_rate1, #use this
        policy_rate2,
        policy_rate3,
        fx, #use this
        cds, #use this
        liquidity, #use this
        gdp_growth, #use this
    )


vix, policy_rate1, policy_rate2, policy_rate3, fx, cds, liquidity, gdp_growth, = dataF()

scopes = [
    'https://www.googleapis.com/auth/spreadsheets']

creds = Credentials.from_service_account_file(
    'credential.json', scopes=scopes)

client = gspread.authorize(creds)

# 1PP6gpBcoOHjjgCx7LuHLa3dv4ET6ufKvpSY4UDvBczQ
sheet_id = '11Ora6_5EoQJdgnUpjjZgFZyrILguo1c32mde_uQwupw'
sheet = client.open_by_key(sheet_id)

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 40]
})

set_with_dataframe(sheet.sheet1, df) 

values_list = sheet.sheet1.get_all_values()
print(values_list)

worksheet_list = map(lambda x: x.title, sheet.worksheets())
worksheet_name = {'vix_data': vix, 'policy_rate1': policy_rate1, 'fx': fx, 'cds': cds, 'liquidity': liquidity, 'gdp_growth': gdp_growth}
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet1 = sheet.worksheet(new_worksheet_name)
    else:
        sheet1 = sheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)
    set_with_dataframe(sheet1, data)

# sheet.worksheet(new_worksheet_name) -- to select worksheet page
# sheet.add_worksheet(new_worksheet_name, rows=10, cols=10) -- add new worksheet page

c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:161: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"]) + pd.offsets.MonthEnd(0)
c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:296: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2["Date"] = pd.to_datetime(df2["Date"]) + pd.offsets.MonthEnd(0)
c:\USERS\AHMADAIZUDEEN\ONEDRIVE - THE SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\DESKTOP\NLP PROJECT\VENC\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a p

[['Name', 'Age'], ['Alice', '25'], ['Bob', '30'], ['Charlie', '40']]


In [4]:
x = [1, 2, 3,4,5]
x[-2:]

[4, 5]

In [124]:
import pandas as pd 

yield_data1 = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\00 dlx.xlsx", header=1)
yield_data1 = yield_data1[[*yield_data1.columns[[0,1,3]]]]
yield_data1.drop([0,1,2], inplace=True)
yield_data1.rename(columns={'.DESC': 'Date',
                          'Indonesia: 10 Year Treasury Bond Mid Yield (% p.a.)': 'Indonesia',
                          'Philippines: 10 Year Treasury Bond Mid Yield (% p.a.)': 'Philippines'},
                    inplace=True)

# yield_data1["Date"] = pd.to_datetime(yield_data1["Date"], errors='coerce') + pd.offsets.MonthEnd(0)
yield_data1['Month'] = yield_data1['Date'].apply(lambda x: x[:3])
yield_data1['Year'] = yield_data1['Date'].apply(lambda x: x[-2:])
yield_data1["Date"] = yield_data1.apply(lambda x: pd.to_datetime(f"{x['Month']} {x['Year']}", format='%b %y') + pd.offsets.MonthEnd(0), axis=1)
yield_data1.drop(['Month', 'Year'], axis=1, inplace=True)
yield_data2 = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx"
                            , sheet_name='Yields2'
                            , header=1)
yield_data2=yield_data2[yield_data2.columns[:11]]
yield_data2.drop([0,1], inplace=True)
yield_data2.rename(columns={'Region': 'Date'}, inplace=True)
yield_data2["Date"] = pd.to_datetime(yield_data2["Date"]) + pd.offsets.MonthEnd(0)
yield_data2.drop('Philippines', axis=1, inplace=True)

yield_data = yield_data2.join(yield_data1.set_index('Date'), on='Date')
yield_data.set_index('Date', inplace=True)

yield_data_calc = yield_data.apply(lambda x : x - x['United States'], axis =1)
yield_data_calc.drop('United States', axis=1, inplace=True)
yield_data_calc = yield_data_calc.loc[yield_data_calc.index >= "2010-01-31"]
yield_data_calc['Asia'] = yield_data_calc.mean(axis= 1)
yield_data_calc = (yield_data_calc - yield_data_calc.mean()) / yield_data_calc.std()
yield_data_calc

,Thailand,Taiwan,Singapore,South Korea,Malaysia,India,Hong Kong SAR (China),China,Indonesia,Philippines,Asia
Date,,,,,,,,,,,
2010-01-31,0.175651,-1.158241,-1.901368,1.572966,-1.004538,-1.107842,-1.050142,-0.741576,NaN,NaN,-1.540744
2010-02-28,0.087742,-1.236897,-1.485320,1.555688,-0.980961,-0.798749,-1.246005,-0.869708,NaN,NaN,-1.505457
2010-03-31,0.046732,-1.259095,-1.260817,1.023806,-1.166129,-0.865235,-1.426455,-0.900146,NaN,NaN,-1.638070
2010-04-30,-0.370188,-1.407762,-1.865981,0.854675,-1.463067,-1.028513,-1.546308,-1.064175,NaN,NaN,-1.940724
2010-05-31,-0.012640,-0.936574,-0.681719,1.458693,-0.888538,-0.862325,-1.134185,-0.777569,NaN,NaN,-1.390754
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,-2.132792,-1.681953,-2.281407,-1.879522,-2.484187,-2.197072,-2.174949,-2.542791,-1.683475,-0.423165,-2.454110
2024-08-31,-1.785504,-1.277460,-1.831526,-1.619181,-1.911162,-1.901352,-1.637959,-2.191315,-1.594144,-0.228112,-2.039476
2024-09-30,-1.675572,-1.240222,-1.707193,-1.427541,-1.779653,-1.885661,-1.585829,-2.078945,-1.606654,-0.250560,-1.949901


In [20]:
import pandas as pd
import numpy as np

key_dict = {
    'Select this link and click Refresh/Edit Download to update data and add or remove series': 'Date',
    'Index: Shenzhen Stock Exchange: Composite': 'China',
    'Equity Market Index: Month End: Hang Seng': 'Hong Kong SAR (China)',
    'Equity Market Index: Month End: BSE: Sensitive 30 (Sensex)': 'India',
    'Equity Market Index: Month End: Jakarta Composite': 'Indonesia',
    'Equity Market Index: Month End: FTSE Bursa Malaysia: Composite': 'Malaysia',
    'Equity Market Index: Month End: PSEi': 'Philippines',
    'Equity Market Index: Month End: FTSE Strait Times': 'Singapore',
    'Equity Market Index: Month End: KOSPI': 'South Korea',
    'TWSE: Equity Market Index: TAIEX Capitalization Weighted: Month Avg': 'Taiwan',
    'Equity Market Index: Month End: SET': 'Thailand',
}


stock_data = pd.read_excel(r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx', sheet_name='Stock Price Indexes')
stock_data = stock_data[key_dict.keys()]
stock_data.drop([0,1,2], inplace=True)
stock_data.rename(columns=key_dict, inplace=True)
stock_data2 = stock_data.copy()
stock_data2["Date"] = pd.to_datetime(stock_data2["Date"]) + pd.offsets.MonthEnd(0)
stock_data2.set_index('Date', inplace=True)
stock_data2 = stock_data2.loc[stock_data2.index >= "2008-12-01"]
stock_data2 = stock_data2.astype(float)

stock_data_calc = stock_data2.apply(lambda x : np.log(x))
stock_data_calc = (stock_data_calc.shift(1) - stock_data_calc) * 100 ## can use pandas .diff() function
stock_data_calc['Asia'] = stock_data_calc.mean(axis=1)
stock_data_calc2 = (stock_data_calc - stock_data_calc.mean())/stock_data_calc.std()
stock_data_calc2 = stock_data_calc2.loc[stock_data_calc2.index >= "2010-01-01"]
stock_data_calc2.tail()

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2024-07-31,0.147440,0.385874,-0.433636,-0.415141,-0.614549,-0.511871,-0.732120,0.287184,-0.661738,-0.196750,-0.294412
2024-08-31,0.639594,-0.588131,0.066133,-1.072737,-0.944027,-0.702972,0.174591,0.819880,1.777759,-0.483978,0.004803
2024-09-30,-2.807292,-2.687516,-0.232705,0.627905,0.684589,-0.941670,-0.827986,0.723699,0.170814,-1.247176,-1.232273
2024-10-31,-0.340054,0.690108,1.361714,0.058893,1.040642,0.499770,0.256581,0.384197,-1.054618,-0.124130,0.360026
2024-11-30,-0.075643,0.784297,0.112344,1.629652,0.255725,1.697787,-1.030216,0.915370,0.394268,0.703304,0.721909


In [84]:
stock_data_calc

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2009-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-02-28,-7.691401,3.577568,5.817696,3.605312,-0.700801,-2.549559,9.080438,8.911400,-0.038691,1.419704,2.143167
2009-03-31,-17.368482,-5.795633,-8.789365,-10.939027,2.055403,-5.910844,-6.383014,-12.640134,-9.557833,0.004635,-7.532430
2009-04-30,-5.566340,-13.388830,-16.089661,-18.341179,-12.703218,-5.736929,-12.184864,-12.681881,-15.022764,-13.058093,-12.477376
2009-05-31,-6.175595,-15.763362,-24.885108,-10.674216,-5.246799,-12.740200,-19.300233,-1.918873,-14.026072,-13.082022,-12.381248
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,0.451555,2.133428,-3.369780,-2.684433,-2.206791,-3.180070,-3.628170,0.974416,-3.460236,-1.518059,-1.648814
2024-08-31,4.219313,-3.648313,-0.761007,-5.561627,-3.222073,-4.120693,0.377164,3.540498,6.157880,-2.851760,-0.587062
2024-09-30,-22.168805,-16.110363,-2.320932,1.879221,1.796479,-5.295591,-4.051651,3.077178,-0.177764,-6.395569,-4.976780


In [ ]:
# China	Hong Kong SAR (China)	India	Indonesia	Malaysia	Philippines	Singapore	South Korea	Taiwan	Thailand

country_list = ['Region', 'China', 'Hong Kong SAR (China)', 'India', 'Indonesia', 'Malaysia', 'Philippines', 'Singapore', 'South Korea', 'Taiwan', 'Thailand']
fx = pd.read_excel(r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx', sheet_name='FX', header=1)
fx.drop([0,1], inplace=True)
fx = fx[country_list]
fx.rename(columns={'Region': 'Date'}, inplace=True)
fx["Date"] = pd.to_datetime(fx["Date"]) + pd.offsets.MonthEnd(0) 
fx.set_index('Date', inplace=True) 
fx = fx.loc[fx.index >= "2009-12-31"]
fx = fx.astype(float)
fx = ((fx / fx.shift(1))-1) * 100
fx['Asia'] = fx.mean(axis=1)
fx = (fx - fx.mean())/fx.std()

ora = pd.read_excel(r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx', sheet_name='ORA', header=1)
ora.drop([0,1], inplace=True)
ora = ora[country_list]
ora.rename(columns={'Region': 'Date'}, inplace=True)
ora["Date"] = pd.to_datetime(ora["Date"]) + pd.offsets.MonthEnd(0)
ora.set_index('Date', inplace=True)
ora = ora.loc[ora.index >= "2009-12-31"]
ora = ora.astype(float)
ora = ora/1000
ora = ((ora.shift(1) / ora)-1) * 100
ora['Asia'] = ora.mean(axis=1)
ora = (ora - ora.mean())/ora.std()


empi = fx - ora
empi

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2009-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-31,0.345952,1.054376,-1.222844,1.503236,-0.576233,0.767593,0.259894,-0.245190,-0.443412,0.880671,0.240604
2010-02-28,0.141868,0.614923,-1.090945,-0.310978,0.488664,0.164352,0.537979,-0.474865,1.017381,-0.142971,-0.016566
2010-03-31,0.552546,-0.614600,-1.371011,-0.122339,-2.360757,-2.002335,1.003273,-0.574068,-0.071055,-0.558462,-0.914192
2010-04-30,1.206054,-0.020688,-1.127284,2.507447,-1.729082,-1.064146,-0.033373,1.018722,-0.401503,0.544436,0.193751
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,0.755658,0.216318,1.329120,0.815858,-0.077479,-0.031528,0.317043,0.104054,-0.095159,0.395229,0.691954
2024-08-31,0.322714,-0.749756,0.422593,-1.428077,-2.242463,-1.512296,-1.513595,-0.905395,0.404154,-1.751240,-1.355650
2024-09-30,-0.805028,-0.818792,1.345380,-1.195530,-1.016961,0.190530,-0.868812,-0.061582,-1.571829,-1.265495,-0.738986


In [21]:
import pandas as pd
import numpy as np

key_dict_fb = {
            'Select this link and click Refresh/Edit Download to update data and add or remove series': 'Date',
            'Index: Shenzhen Stock Exchange: Financials': 'China',
            'Index: Hang Seng Finance': 'Hong Kong SAR (China)',
            'BSE: Index: Bankex': 'India',
            '(DC)IDX: Index: Finance': 'Indonesia1',
            'IDX: Index: Financials': 'Indonesia2',
            'Bursa Malaysia: Index: Financial Services': 'Malaysia',
            'Index: BankingFinancial Services': 'Philippines',
            'Equity Market Index: Month End: FTSE Strait Times':'Singapore1',
            'Singapore Exchange Turnover: Value: Mainboard': 'Singapore2',
            'Singapore Exchange Turnover: Value: Mainboard: Financials': 'Singapore3',
            'Index: KOSPI: Financial Institutions': 'South Korea',
            'TWSE: Equity Market Index: Finance and Insurance': 'Taiwan',
            'SET: Index: Banking': 'Thailand1', #T
            'SET: Index: Finance and Securities': 'Thailand2', #V
            'SET: Index: Insurance': 'Thailand3', #U
            'SET: Market Capitalization: Banking': 'Thailand4', #X
            'SET: Market Capitalization: Finance and Securities': 'Thailand5', #Y
            'SET: Market Capitalization: Insurance': 'Thailand6', #Z
            'Financial Index': 'Financial Index1',
            }
fb = pd.read_excel(r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx', sheet_name='Stock Price Indexes')
fb = fb[key_dict_fb.keys()]
fb.drop([0,1,2], inplace=True)
fb.rename(columns=key_dict_fb, inplace=True)
fb.reset_index(drop=True, inplace=True)
fb['Date'] = pd.to_datetime(fb['Date']) + pd.offsets.MonthEnd(0) 
fb.tail()

,Date,China,Hong Kong SAR (China),India,Indonesia1,Indonesia2,Malaysia,Philippines,Singapore1,Singapore2,Singapore3,South Korea,Taiwan,Thailand1,Thailand2,Thailand3,Thailand4,Thailand5,Thailand6,Financial Index1
294,2024-07-31,990.09,30406.6,58865.99,NaN,1401.87,18173.08,1990.27,3455.94,25351.1,13812.2,482.82,2076.29,356.53,2504.22,7618.09,1921.180341,650.205801,294.074227,124.092973
295,2024-08-31,956.38,31092.9,58311.51,NaN,1473.486,19717.53,2110.63,3442.93,27812.5,15595.9,472.03,2052.59,374.34,2761.16,7770.39,1921.180341,650.205801,294.074227,123.831017
296,2024-09-30,1386.99,35263.86,60038.09,NaN,1521.295,19289.18,2297.62,3585.29,29314.5,17531.9,466.36,2082.29,395.82,3140.94,9512.01,1921.180341,650.205801,294.074227,126.893232
297,2024-10-31,1454.15,34590.61,58663.55,NaN,1524.625,19086.68,2331.87,3558.88,25283.2,14243.8,475.39,2079.72,391.61,3084.32,9630.81,1921.180341,650.205801,294.074227,126.366638
298,2024-11-30,1560.77,33355.44,59298.07,NaN,1455.971,19190.03,2254.66,3739.29,29017.7,17856.4,481.37,2071.79,390.72,2924.16,8927.12,1921.180341,650.205801,294.074227,130.308583


In [22]:
import warnings
warnings.filterwarnings("ignore")

fb_sg = fb[['Date', 'Singapore1', 'Singapore2', 'Singapore3']]
fb_sg['Date'] = pd.to_datetime(fb_sg['Date']) + pd.offsets.MonthEnd(0) 
fb_sg['Nonfinancials'] = fb_sg['Singapore2'] - fb_sg['Singapore3']
fb_sg['weigths_financial'] = fb_sg['Singapore3'] / fb_sg['Singapore2']
fb_sg['weigths_nonfinancial'] = fb_sg['Nonfinancials'] / fb_sg['Singapore2']
fb_sg['Price_Growth'] = ((fb_sg['Singapore1'] / fb_sg['Singapore1'].shift(1))-1) * 100
fb_sg['Financial_Growth'] = fb_sg['Price_Growth'] * fb_sg['weigths_financial']
fb_sg['Financial Index - SG'] = np.nan
fb_sg.reset_index(drop=True, inplace=True)
fb_sg.at[0, 'Financial Index - SG'] = 100

set_base_sg = pd.to_datetime('2008-12-01') + pd.offsets.MonthEnd(0)
for i in range(1, len(fb_sg)):
    if set_base_sg is not None:
        if fb_sg.at[i, 'Date'] == set_base_sg: # set when the base date 
            fb_sg.at[i, 'Financial Index - SG'] = 100
        else:
            fb_sg.at[i, 'Financial Index - SG'] = fb_sg.at[i-1, 'Financial Index - SG'] * ((fb_sg.at[i, 'Financial_Growth'] / 100) + 1)
    else:
        fb_sg.at[i, 'Financial Index - SG'] = fb_sg.at[i-1, 'Financial Index - SG'] * ((fb_sg.at[i, 'Financial_Growth'] / 100) + 1)
fb_sg.tail()

,Date,Singapore1,Singapore2,Singapore3,Nonfinancials,weigths_financial,weigths_nonfinancial,Price_Growth,Financial_Growth,Financial Index - SG
294,2024-07-31,3455.94,25351.1,13812.2,11538.9,0.544836,0.455164,3.694791,2.013056,124.092973
295,2024-08-31,3442.93,27812.5,15595.9,12216.6,0.560751,0.439249,-0.376453,-0.211097,123.831017
296,2024-09-30,3585.29,29314.5,17531.9,11782.6,0.598062,0.401938,4.13485,2.472898,126.893232
297,2024-10-31,3558.88,25283.2,14243.8,11039.4,0.56337,0.43663,-0.736621,-0.41499,126.366638
298,2024-11-30,3739.29,29017.7,17856.4,11161.3,0.615362,0.384638,5.069291,3.119451,130.308583


In [23]:
import warnings
warnings.filterwarnings("ignore")

fb_th = fb[['Date', 'Thailand1', 'Thailand2', 'Thailand3', 'Thailand4', 'Thailand5', 'Thailand6']]
fb_th['Date'] = pd.to_datetime(fb_th['Date']) + pd.offsets.MonthEnd(0) 
fb_th['growth_banking'] = ((fb_th['Thailand1'] / fb_th['Thailand1'].shift(1))-1) * 100
fb_th['growth_finance'] = ((fb_th['Thailand2'] / fb_th['Thailand2'].shift(1))-1) * 100
fb_th['growth_insurance'] = ((fb_th['Thailand3'] / fb_th['Thailand3'].shift(1))-1) * 100   
fb_th['weight_banking'] = fb_th['Thailand4'] / (fb_th['Thailand4'] + fb_th['Thailand5'] + fb_th['Thailand6'])
fb_th['weight_finance'] = fb_th['Thailand5'] / (fb_th['Thailand4'] + fb_th['Thailand5'] + fb_th['Thailand6'])
fb_th['weight_insurance'] = fb_th['Thailand6'] / (fb_th['Thailand4'] + fb_th['Thailand5'] + fb_th['Thailand6'])
fb_th['financial_growth'] = fb_th['growth_banking'] * fb_th['weight_banking']  + fb_th['growth_finance'] * fb_th['weight_finance'] + fb_th['growth_insurance'] * fb_th['weight_insurance']
fb_th['Financial Index - TH'] = np.nan
fb_th.reset_index(drop=True, inplace=True)
fb_th.at[0, 'Financial Index - TH'] = 100

set_base_th = None
for i in range(1, len(fb_th)):
    if set_base_th is not None:
        if fb_th.at[i, 'Date'] == pd.to_datetime('2008-12-01') + pd.offsets.MonthEnd(0): # set when the base date 
            fb_th.at[i, 'Financial Index - TH'] = 100
        else:
            fb_th.at[i, 'Financial Index - TH'] = fb_th.at[i-1, 'Financial Index - TH'] * ((fb_th.at[i, 'financial_growth'] / 100) + 1)
    else:
        fb_th.at[i, 'Financial Index - TH'] = fb_th.at[i-1, 'Financial Index - TH'] * ((fb_th.at[i, 'financial_growth'] / 100) + 1)

fb_th.tail()

,Date,Thailand1,Thailand2,Thailand3,Thailand4,Thailand5,Thailand6,growth_banking,growth_finance,growth_insurance,weight_banking,weight_finance,weight_insurance,financial_growth,Financial Index - TH
294,2024-07-31,356.53,2504.22,7618.09,1921.180341,650.205801,294.074227,1.497424,-8.752305,-3.485772,0.670461,0.226911,0.102627,-1.339769,151.078633
295,2024-08-31,374.34,2761.16,7770.39,1921.180341,650.205801,294.074227,4.995372,10.260281,1.999189,0.670461,0.226911,0.102627,5.88255,159.965910
296,2024-09-30,395.82,3140.94,9512.01,1921.180341,650.205801,294.074227,5.738099,13.754364,22.413547,0.670461,0.226911,0.102627,9.268436,174.792248
297,2024-10-31,391.61,3084.32,9630.81,1921.180341,650.205801,294.074227,-1.063615,-1.802645,1.248947,0.670461,0.226911,0.102627,-0.993977,173.054853
298,2024-11-30,390.72,2924.16,8927.12,1921.180341,650.205801,294.074227,-0.227267,-5.192717,-7.306654,0.670461,0.226911,0.102627,-2.080522,169.454408


In [24]:
# fb_id = fb[['Date', 'Indonesia1', 'Indonesia2']] ##256
fb_id_index = fb['Indonesia1'].isna().idxmax()
fb_id1 = fb[['Date', 'Indonesia1']]
fb_id1 = fb_id1[fb_id1.index < fb_id_index]
fb_id2 = fb[['Date', 'Indonesia2']]
fb_id2 = fb_id2[fb_id2.index >= fb_id_index]
fb_id1.rename(columns={'Indonesia1': 'Indonesia'}, inplace=True)
fb_id2.rename(columns={'Indonesia2': 'Indonesia'}, inplace=True)
fb_id = pd.concat([fb_id1, fb_id2])
fb_id
# fb_id.iloc[fb_id_ind-10:fb_id_ind+10]

,Date,Indonesia
0,2000-01-31,56.886
1,2000-02-29,56.14
2,2000-03-31,56.456
3,2000-04-30,50.554
4,2000-05-31,42.52
...,...,...
294,2024-07-31,1401.87
295,2024-08-31,1473.486
296,2024-09-30,1521.295
297,2024-10-31,1524.625


In [25]:
fb_compile = pd.concat([fb[['Date', 'China', 'Hong Kong SAR (China)', 'India']], fb_id[['Indonesia']], fb[['Malaysia', 'Philippines']], fb_sg[['Financial Index - SG']], fb[['South Korea', 'Taiwan']], fb_th[['Financial Index - TH']]], axis=1)
fb_compile.set_index('Date', inplace=True)
fb_compile.rename(columns={'Financial Index - SG': 'Singapore', 'Financial Index - TH': 'Thailand'}, inplace=True)
fb_compile = fb_compile.astype(float)
fb_compile.tail()

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand
Date,,,,,,,,,,
2024-07-31,990.09,30406.60,58865.99,1401.870,18173.08,1990.27,124.092973,482.82,2076.29,151.078633
2024-08-31,956.38,31092.90,58311.51,1473.486,19717.53,2110.63,123.831017,472.03,2052.59,159.965910
2024-09-30,1386.99,35263.86,60038.09,1521.295,19289.18,2297.62,126.893232,466.36,2082.29,174.792248
2024-10-31,1454.15,34590.61,58663.55,1524.625,19086.68,2331.87,126.366638,475.39,2079.72,173.054853
2024-11-30,1560.77,33355.44,59298.07,1455.971,19190.03,2254.66,130.308583,481.37,2071.79,169.454408


In [26]:
fb_compile_calc = fb_compile.apply(lambda x : np.log(x))
fb_compile_calc = (fb_compile_calc.shift(1) - fb_compile_calc) * 100
fb_compile_calc['Asia'] = fb_compile_calc.mean(axis=1)
fb_compile_calc.tail()

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2024-07-31,-3.996638,3.314611,1.307808,-2.667681,-4.033071,-3.355735,-1.993062,-5.172724,-3.172679,1.348825,-1.842035
2024-08-31,3.464052,-2.231981,0.946400,-4.982396,-8.156671,-5.871617,0.211320,2.260137,1.148024,-5.716028,-1.892876
2024-09-30,-37.173589,-12.587915,-2.917969,-3.193092,2.196377,-8.488732,-2.442817,1.208467,-1.436584,-8.863739,-7.369959
2024-10-31,-4.728561,1.927639,2.316061,-0.218653,1.055361,-1.479672,0.415854,-1.917765,0.123498,0.998950,-0.150729
2024-11-30,-7.075775,3.636138,-1.075818,4.607545,-0.540016,3.367134,-3.071785,-1.250069,0.382030,2.102470,0.108185


In [27]:
# stock_data_calc.head()
# fb_compile_calc
covariance = pd.DataFrame(index=stock_data_calc.index, columns=stock_data_calc.columns)
date_dict = {k:v for k,v in enumerate(stock_data_calc.index)}
for x in stock_data_calc.columns:
    for i in range(1, len(stock_data_calc)-12):
        covariance.at[date_dict[i+12], x] = np.cov(stock_data_calc.loc[date_dict[i]:date_dict[i+12], x].values, fb_compile_calc.loc[date_dict[i]:date_dict[i+12], x].values, bias=True)[0, 1]
covariance

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2008-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,20.533558,19.996926,10.027418,6.184568,4.016233,14.752308,5.303552,13.368511,6.943452,8.498342,4.999853
2024-08-31,17.812129,20.195331,10.115921,7.681792,3.443724,15.772267,4.406056,14.744421,9.067145,9.637089,4.261046
2024-09-30,82.970547,28.193614,8.371674,7.073583,4.136624,14.960788,4.138152,15.452678,6.844371,14.514265,5.520723


In [28]:
variance = pd.DataFrame(index=stock_data_calc.index, columns=stock_data_calc.columns)

for x in stock_data_calc.columns:
    for i in range(1, len(stock_data_calc)-12):
        variance.at[date_dict[i+12], x] = np.var(stock_data_calc.loc[date_dict[i]:date_dict[i+12], x].values, ddof=1)
variance.tail()

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2024-07-31,38.378925,27.498703,10.371686,6.162296,4.584507,14.752316,10.382068,29.301216,7.885925,9.612806,6.836588
2024-08-31,38.001822,25.351785,10.358538,7.238465,3.123642,15.771782,8.553454,30.138342,13.422233,9.187675,6.084463
2024-09-30,81.366707,40.010577,8.754836,7.778772,3.590137,12.980209,7.751594,30.209,11.998984,13.350041,6.521484
2024-10-31,81.84855,40.604776,13.733661,7.720868,4.128173,13.55272,7.809213,29.211244,11.955576,10.623302,6.285684
2024-11-30,81.450353,41.045958,12.071723,10.677602,4.257421,16.11833,6.096291,25.075554,11.849512,7.857192,5.275511


In [ ]:
financial_sector_beta = covariance / variance
financial_sector_beta = (financial_sector_beta - financial_sector_beta.mean()) / financial_sector_beta.std()
financial_sector_beta = financial_sector_beta.applymap(lambda x: x if x > 1 else 0)
financial_sector_beta.tail()

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2024-07-31,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0
2024-08-31,0.0,0.0,0.0,0.0,1.131309,0.000000,0.000000,0.0,0.0,0.000000,0.0
2024-09-30,0.0,0.0,0.0,0.0,1.435961,1.614732,1.141560,0.0,0.0,0.000000,0.0
2024-10-31,0.0,0.0,0.0,0.0,0.000000,1.355985,1.143682,0.0,0.0,1.096153,0.0
2024-11-30,0.0,0.0,0.0,0.0,0.000000,0.000000,1.278805,0.0,0.0,1.573340,0.0


In [1]:
from data import Data

df = Data()
yield_data_calc, stock_data_calc2, empi, financial_sector_beta = df.financial_stress_index()

In [2]:
empi

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2009-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-31,0.345952,1.054376,-1.222844,1.503236,-0.576233,0.767593,0.259894,-0.245190,-0.443412,0.880671,0.240604
2010-02-28,0.141868,0.614923,-1.090945,-0.310978,0.488664,0.164352,0.537979,-0.474865,1.017381,-0.142971,-0.016566
2010-03-31,0.552546,-0.614600,-1.371011,-0.122339,-2.360757,-2.002335,1.003273,-0.574068,-0.071055,-0.558462,-0.914192
2010-04-30,1.206054,-0.020688,-1.127284,2.507447,-1.729082,-1.064146,-0.033373,1.018722,-0.401503,0.544436,0.193751
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,0.755658,0.216318,1.329120,0.815858,-0.077479,-0.031528,0.317043,0.104054,-0.095159,0.395229,0.691954
2024-08-31,0.322714,-0.749756,0.422593,-1.428077,-2.242463,-1.512296,-1.513595,-0.905395,0.404154,-1.751240,-1.355650
2024-09-30,-0.805028,-0.818792,1.345380,-1.195530,-1.016961,0.190530,-0.868812,-0.061582,-1.571829,-1.265495,-0.738986


In [3]:
empi

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2010-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-02-28,0.143484,0.620214,-1.096159,-0.302058,0.484804,0.167403,0.537833,-0.475171,1.014575,-0.139063,-0.015662
2010-03-31,0.553188,-0.608298,-1.374987,-0.104662,-2.359402,-2.000556,1.002212,-0.577483,-0.074869,-0.549396,-0.911197
2010-04-30,1.205138,-0.014577,-1.131913,2.537275,-1.730153,-1.053503,-0.031630,1.016182,-0.405733,0.552196,0.196867
2010-05-31,-1.768486,1.309007,0.132747,-1.600287,0.423346,1.908928,-0.410022,-1.061048,1.960044,-1.327648,-0.273737
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,0.755772,0.221448,1.320606,0.833029,-0.081068,-0.025238,0.317714,0.102838,-0.095830,0.403858,0.693724
2024-08-31,0.324017,-0.744723,0.415295,-1.406721,-2.243530,-1.504455,-1.507795,-0.909418,0.398560,-1.738309,-1.351572
2024-09-30,-0.800599,-0.811740,1.337039,-1.186890,-1.019955,0.204355,-0.864839,-0.064444,-1.573772,-1.252276,-0.735632


In [ ]:
from arch import arch_model

test_df=financial_sector_beta['China']
am = arch_model(test_df, mean='AR', lags=1, vol='GARCH', p=1, q=1)

In [ ]:
# len(result.conditional_volatility)
# result.fit_start

from arch import arch_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Obtain Equity Market Data and Calculate Returns
# For this example, we'll create some dummy return data
np.random.seed(0)
n_obs = 1000
returns = pd.Series(np.random.normal(0, 0.01, n_obs))
returns.name = 'Equity Returns'

# In a real application, you would load your actual stock price data,
# convert it to month-on-month returns by taking the difference between
# current and previous month’s stock price index in natural logarithm form.
# For example:
# prices = pd.read_csv('stock_prices.csv', index_col='Date', parse_dates=True)
# returns = np.log(prices['Price'] / prices['Price'].shift(1)).dropna()

# 2. Specify and Fit the GARCH(1,1) Model with an AR(1) Mean Model
# The study's mean equation is y_t = alpha + beta*y_{t-1} + epsilon_t, which is an AR(1) process.
# We will specify this in the arch_model.
am = arch_model(returns, mean='AR', lags=1, vol='GARCH', p=1, q=1)

# Fit the model. disp='off' suppresses the fitting output.
res = am.fit(disp='off')

# 3. Extract Conditional Volatility
# The conditional volatility (sigma_t) can be accessed from the results.
conditional_volatility = res.conditional_volatility

# You can also access the conditional variance (sigma^2_t)
conditional_variance = res.conditional_volatility**2

# 4. Visualise the Returns and Conditional Volatility (Optional)
plt.figure(figsize=(10, 6))
plt.plot(returns, label='Equity Returns')
plt.plot(conditional_volatility, label='Conditional Volatility (GARCH(1,1))', color='red')
plt.title('Equity Returns and GARCH(1,1) Conditional Volatility')
plt.xlabel('Time')
plt.ylabel('Returns / Volatility')
plt.legend()
plt.show()

0

In [32]:
import pandas as pd
import numpy as np

key_dict = {
    'Select this link and click Refresh/Edit Download to update data and add or remove series': 'Date',
    'Index: Shenzhen Stock Exchange: Composite': 'China',
    'Equity Market Index: Month End: Hang Seng': 'Hong Kong SAR (China)',
    'Equity Market Index: Month End: BSE: Sensitive 30 (Sensex)': 'India',
    'Equity Market Index: Month End: Jakarta Composite': 'Indonesia',
    'Equity Market Index: Month End: FTSE Bursa Malaysia: Composite': 'Malaysia',
    'Equity Market Index: Month End: PSEi': 'Philippines',
    'Equity Market Index: Month End: FTSE Strait Times': 'Singapore',
    'Equity Market Index: Month End: KOSPI': 'South Korea',
    'TWSE: Equity Market Index: TAIEX Capitalization Weighted: Month Avg': 'Taiwan',
    'Equity Market Index: Month End: SET': 'Thailand',
}


stock_data = pd.read_excel(r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx', sheet_name='Stock Price Indexes')
stock_data = stock_data[key_dict.keys()]
stock_data.drop([0,1,2], inplace=True)
stock_data.rename(columns=key_dict, inplace=True)
stock_data2 = stock_data.copy()
stock_data2["Date"] = pd.to_datetime(stock_data2["Date"]) + pd.offsets.MonthEnd(0)
stock_data2.set_index('Date', inplace=True)
stock_data2 = stock_data2.loc[stock_data2.index >= "2008-12-01"]
stock_data2 = stock_data2.astype(float)

stock_data_calc = stock_data2.apply(lambda x : np.log(x))
stock_data_calc = (stock_data_calc.shift(1) - stock_data_calc) * 100 ## can use pandas .diff() function
stock_data_calc = stock_data_calc.loc[stock_data_calc.index >= "2010-01-01"]
stock_data_calc['Asia'] = stock_data_calc.mean(axis=1)
stock_data_calc2 = (stock_data_calc - stock_data_calc.mean())/stock_data_calc.std()
# stock_data_calc2 = stock_data_calc.apply(lambda x: (x - x.mean()) / x.std(), axis=0)
# stock_data_calc2 = stock_data_calc2.loc[stock_data_calc2.index >= "2010-01-01"]

In [33]:
stock_data_calc2

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2010-01-31,0.985631,1.429724,1.552245,-0.598928,0.411379,0.774487,1.382779,1.109915,-0.729120,1.270314,1.194631
2010-02-28,-0.575225,-0.424380,0.086492,0.742925,-0.271396,-0.535273,-0.014485,0.152743,2.491955,-0.699817,0.000402
2010-03-31,-0.403999,-0.532226,-1.179834,-2.000632,-1.272603,-0.697535,-1.173968,-1.254744,-1.062781,-1.891677,-1.515336
2010-04-30,1.191524,0.095233,0.141128,-1.543941,-0.619549,-0.733150,-0.706569,-0.571045,-0.787946,0.788312,-0.194366
2010-05-31,1.041462,1.124629,0.925268,1.655979,1.640164,0.198716,1.971419,1.336176,1.996414,0.469338,1.663341
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,0.100592,0.357129,-0.529081,-0.527123,-0.712385,-0.568283,-0.869939,0.257865,-0.776042,-0.256413,-0.395732
2024-08-31,0.612060,-0.641776,0.018428,-1.246655,-1.059951,-0.762792,0.129685,0.815971,1.831668,-0.554602,-0.071503
2024-09-30,-2.970088,-2.794832,-0.308957,0.614162,0.658069,-1.005747,-0.975629,0.715202,0.113918,-1.346927,-1.411999


In [31]:
stock_data_calc

,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Asia
Date,,,,,,,,,,,
2008-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-31,-9.800571,8.023404,2.339403,1.692032,-0.874409,2.583203,0.860317,-3.292550,0.473542,2.764779,0.476915
2009-02-28,-7.691401,3.577568,5.817696,3.605312,-0.700801,-2.549559,9.080438,8.911400,-0.038691,1.419704,2.143167
2009-03-31,-17.368482,-5.795633,-8.789365,-10.939027,2.055403,-5.910844,-6.383014,-12.640134,-9.557833,0.004635,-7.532430
2009-04-30,-5.566340,-13.388830,-16.089661,-18.341179,-12.703218,-5.736929,-12.184864,-12.681881,-15.022764,-13.058093,-12.477376
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,0.451555,2.133428,-3.369780,-2.684433,-2.206791,-3.180070,-3.628170,0.974416,-3.460236,-1.518059,-1.648814
2024-08-31,4.219313,-3.648313,-0.761007,-5.561627,-3.222073,-4.120693,0.377164,3.540498,6.157880,-2.851760,-0.587062
2024-09-30,-22.168805,-16.110363,-2.320932,1.879221,1.796479,-5.295591,-4.051651,3.077178,-0.177764,-6.395569,-4.976780


In [40]:
import requests as r
for x in list1:
    response = r.get(x)
    print(response.status_code)

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


In [1]:
import pandas as pd 

df = pd.read_excel(r'C:\Users\Admin\OneDrive\Desktop\data science project\cfm-nlp\cfm-nlp\OE Inflation_February 2025.xlsx', sheet_name='Default (2)')
df

,country,quarter,inflation2000,inflation2001,inflation2002,inflation2003,inflation2004,inflation2005,inflation2006,inflation2007,...,inflation2013,inflation2014,inflation2015,inflation2016,inflation2017,inflation2018,inflation2019,inflation2020,inflation2021,inflation2022
0,Albania,Q1,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,...,1.85,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40
1,Albania,Q2,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,...,1.85,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40
2,Albania,Q3,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,...,1.85,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40
3,Albania,Q4,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,...,1.85,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40
4,Algeria,Q1,1.66,0.83,4.42,-1.05,6.43,4.53,-0.88,3.11,...,5.28,2.30,5.54,4.70,7.21,1.95,3.58,2.40,4.86,9.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,Vietnam,Q4,NaN,NaN,4.04,3.03,9.68,8.68,6.58,12.62,...,6.03,1.84,0.60,4.72,2.59,2.99,5.24,0.18,1.81,4.54
548,Zambia,Q1,23.47,28.77,18.24,22.75,17.90,17.93,10.99,12.71,...,6.55,7.56,7.06,22.12,6.60,7.16,7.63,14.12,22.80,13.03
549,Zambia,Q2,24.57,20.43,23.43,21.76,18.35,18.97,8.31,11.01,...,7.33,7.95,7.07,20.98,6.64,7.17,8.33,15.64,24.43,9.72
550,Zambia,Q3,27.73,17.54,23.76,20.68,17.41,18.97,8.00,9.18,...,7.20,8.14,8.01,18.98,6.55,7.88,10.45,15.74,22.17,9.95


In [21]:
df2 = df.copy()

df2['index'] = df2['country'] + ' ' + df2['quarter']
df2.drop(['country', 'quarter'], axis=1, inplace=True)
df2
# df.melt(id_vars=['index'], var_name='Date', value_name='Inflation Rate')

,inflation2000,inflation2001,inflation2002,inflation2003,inflation2004,inflation2005,inflation2006,inflation2007,inflation2008,inflation2009,...,inflation2014,inflation2015,inflation2016,inflation2017,inflation2018,inflation2019,inflation2020,inflation2021,inflation2022,index
0,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,2.06,3.54,...,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40,Albania Q1
1,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,2.06,3.54,...,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40,Albania Q2
2,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,2.06,3.54,...,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40,Albania Q3
3,4.20,3.53,2.10,2.87,2.16,2.05,2.55,2.99,2.06,3.54,...,0.66,1.96,2.18,1.80,1.80,1.15,1.05,3.72,7.40,Albania Q4
4,1.66,0.83,4.42,-1.05,6.43,4.53,-0.88,3.11,5.42,9.39,...,2.30,5.54,4.70,7.21,1.95,3.58,2.40,4.86,9.62,Algeria Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,NaN,NaN,4.04,3.03,9.68,8.68,6.58,12.62,19.90,6.51,...,1.84,0.60,4.72,2.59,2.99,5.24,0.18,1.81,4.54,Vietnam Q4
548,23.47,28.77,18.24,22.75,17.90,17.93,10.99,12.71,9.92,13.34,...,7.56,7.06,22.12,6.60,7.16,7.63,14.12,22.80,13.03,Zambia Q1
549,24.57,20.43,23.43,21.76,18.35,18.97,8.31,11.01,11.97,14.31,...,7.95,7.07,20.98,6.64,7.17,8.33,15.64,24.43,9.72,Zambia Q2
550,27.73,17.54,23.76,20.68,17.41,18.97,8.00,9.18,14.12,12.73,...,8.14,8.01,18.98,6.55,7.88,10.45,15.74,22.17,9.95,Zambia Q3


In [ ]:
# df3 = df2['index']+df2[df2.columns[:23]]
# df3 = pd.concat([df2['index'], df2[df2.columns[:23]]], axis=1)
# df3
df3 = df2.melt(id_vars=['index'], var_name='Date', value_name='Inflation Rate')
df3['country'] = df3['index'].apply(lambda x: x.split()[0])
df3['quarter'] = df3['index'].apply(lambda x: x.split()[1])
df3.drop('index', axis=1, inplace=True)
# df3[['country', 'quarter', 'Date', 'Inflation Rate']].to_excel('output.xlsx', index=False)

In [32]:
df3.sort_values(['country', 'quarter', 'Date'], inplace=True)
df3.reset_index(drop=True, inplace=True)
df3

,Date,Inflation Rate,country,quarter
0,inflation2000,4.20,Albania,Q1
1,inflation2001,3.53,Albania,Q1
2,inflation2002,2.10,Albania,Q1
3,inflation2003,2.87,Albania,Q1
4,inflation2004,2.16,Albania,Q1
...,...,...,...,...
12691,inflation2018,5.22,eSwatini,Q4
12692,inflation2019,2.00,eSwatini,Q4
12693,inflation2020,4.60,eSwatini,Q4
12694,inflation2021,3.50,eSwatini,Q4


In [34]:
df3[['country', 'quarter', 'Date', 'Inflation Rate']].to_excel('output.xlsx', index=False)